# Section 1. Install deps




In [1]:
!pip install pandas numpy gdown gensim nltk

# Section 2. Data access


In [14]:
import pandas as pd
import gdown
import ast

file_id = "1Nv079uUCOAsO-cVQQ_UflV5xy9nkgZ6O"

url = f"https://drive.google.com/uc?id={file_id}"

output = "processed_v2.csv"
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)
df["text"] = df["text"].apply(lambda x: ast.literal_eval(x)["clean"])

df.head(5)

Downloading...
From: https://drive.google.com/uc?id=1Nv079uUCOAsO-cVQQ_UflV5xy9nkgZ6O
To: /content/processed_v2.csv
100%|██████████| 4.21M/4.21M [00:00<00:00, 203MB/s]


,title,text,category_id
0,Івано-Франківський драмтеатр знайшов архівні д...,на пресконференції гендиректор-художній керівн...,0
1,Премія Олеся Гончара оголосила номінантів,"Держмистецтв. Цьогоріч на здобуття премії, яку...",0
2,Біографічний фільм «Я граю Роккі» вийде у лист...,The Hollywood Reporter. Стрічка від кіностуді...,0
3,Netflix підписав угоду на трансляцію фільмів S...,"Deadline. Зазначається, що найважливішим момен...",0
4,На Венеційській бієнале Україну представить ск...,розповіли учасники Венеційської бієнале від Ук...,0


# Section 3. Corpus preparation


In [15]:
import pandas as pd
import ast
import re
from gensim.models import Word2Vec, FastText
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

def preparation(df):
    initial_count = len(df)
    df = df.dropna(subset=['text'])
    df = df[df['text'].str.strip() != ""]

    df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
    df = df[df['word_count'] >= 3]

    tokenized_data = [
        [word.lower() for word in word_tokenize(text) if word.isalpha()]
        for text in df['text']
    ]
    num_docs = len(tokenized_data)
    total_tokens = sum(len(doc) for doc in tokenized_data)

    return tokenized_data, num_docs, total_tokens

tokenized_data, num_docs, total_tokens = preparation(df)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Section 4. Tokenization check


In [16]:
print("--- ЗВІТ ПРО КОРПУС ---")
print(f"Кількість документів у корпусі: {num_docs}")
print(f"Приблизна кількість токенів: {total_tokens}")
print(f"Тип тексту: Сирий очищений текст (без лематизації)")
print(f"Метод токенізації: NLTK word_tokenize + фільтрація isalpha()")
print("Робота з формами: Працюємо зі словами у вихідній формі.")
print("Обґрунтування: Вибрано цей варіант для збереження граматичних нюансів контексту.")
print("-----------------------\n")


--- ЗВІТ ПРО КОРПУС ---
Кількість документів у корпусі: 837
Приблизна кількість токенів: 141468
Тип тексту: Сирий очищений текст (без лематизації)
Метод токенізації: NLTK word_tokenize + фільтрація isalpha()
Робота з формами: Працюємо зі словами у вихідній формі.
Обґрунтування: Вибрано цей варіант для збереження граматичних нюансів контексту.
-----------------------



# Section 5. Train Word2Vec


In [17]:
from gensim.models import Word2Vec

params = {
  "vector_size": 100,
  "window": 5,
  "min_count": 3,
  "sg": 1,
  "workers": 4
}

print("Параметри (vector_size=100, window=5, min_count=3)")
print("Вибір архітектури: sg=1 (Skip-gram)")
print("Пояснення: Skip-gram краще працює з рідкісними словами та невеликими корпусами, оскільки намагається передбачити контекст за словом, а не навпаки.")
print("\nТренування Word2Vec (Skip-gram)...")
w2v_model = Word2Vec(sentences=tokenized_data, **params)


Параметри (vector_size=100, window=5, min_count=3)
Вибір архітектури: sg=1 (Skip-gram)
Пояснення: Skip-gram краще працює з рідкісними словами та невеликими корпусами, оскільки намагається передбачити контекст за словом, а не навпаки.

Тренування Word2Vec (Skip-gram)...


# Section 6. Train FastText


In [18]:
from gensim.models import FastText

print("Параметри (vector_size=100, window=5, min_count=3)")
print("Вибір архітектури: sg=1 (Skip-gram)")
print("Пояснення: Skip-gram краще працює з рідкісними словами та невеликими корпусами, оскільки намагається передбачити контекст за словом, а не навпаки.")
print("\nТренування FastText (Skip-gram)...")

ft_model = FastText(sentences=tokenized_data, **params)


Параметри (vector_size=100, window=5, min_count=3)
Вибір архітектури: sg=1 (Skip-gram)
Пояснення: Skip-gram краще працює з рідкісними словами та невеликими корпусами, оскільки намагається передбачити контекст за словом, а не навпаки.

Тренування FastText (Skip-gram)...


# Section 7. Nearest neighbors analysis


In [19]:
import pandas as pd

def analyze_models(w2v_model, ft_model):

    test_words = [
        "фільм",        # Загальне часте слово
        "дискваліфікація",       # Доменний термін
        "євробачення",   # Власна назва (часта)
        "інфляція", # Рідкісне доменне слово
        "фейки",    # Специфічний термін
        "оприлюднений",  # Словоформа 1 (пасивний стан)
        "відновили",    # Словоформа 2 (минулий час, множина)
        "київ",      # Власна назва (локація)
        "Phillip Nova",  # Трансліт / Складна власна назва
        "сценарію"      # Слово з морфологічною варіативністю
    ]

    print(f"{'Термін':<15} | {'Модель':<10} | {'Найближчі сусіди (Nearest Neighbors)'}")
    print("-" * 100)

    for term in test_words:
        print(f"{term:<15} | Word2Vec  | ", end="")
        if term in w2v_model.wv:
            w2v_neighbors = w2v_model.wv.most_similar(term, topn=5)
            print(", ".join([f"{n} ({s:.2f})" for n, s in w2v_neighbors]))
        else:
            print("Слово відсутнє у словнику W2V")

        print(f"{'':<15} | FastText  | ", end="")
        try:
            ft_neighbors = ft_model.wv.most_similar(term, topn=5)
            print(", ".join([f"{n} ({s:.2f})" for n, s in ft_neighbors]))
        except Exception as e:
            print(f"Помилка FastText: {e}")

        print("-" * 100)

analyze_models(w2v_model, ft_model)

Термін          | Модель     | Найближчі сусіди (Nearest Neighbors)
----------------------------------------------------------------------------------------------------
фільм           | Word2Vec  | документальний (0.98), нагороду (0.97), оскар (0.96), номінантів (0.96), повнометражний (0.96)
                | FastText  | фільму (0.99), фільмі (0.99), фільмом (0.99), фільмах (0.99), найкраща (0.98)
----------------------------------------------------------------------------------------------------
дискваліфікація | Word2Vec  | загрожувала (1.00), якому (0.99), українцеві (0.99), намір (0.99), порушив (0.99)
                | FastText  | дискваліфікували (1.00), дискваліфікацію (1.00), кваліфікацію (1.00), дискваліфікації (0.99), кваліфікація (0.99)
----------------------------------------------------------------------------------------------------
євробачення     | Word2Vec  | перемоги (0.99), працювали (0.99), майдані (0.99), київщині (0.99), євробаченні (0.99)
                | FastT

# Section 8. Domain terms analysis


In [20]:
import pandas as pd

def analyze_domain_specific_terms(w2v_model, ft_model):

    domain_terms = [
        "саундтрек",
        "енергосистеми",
        "санкції",
        "законопроєкт",
        "пенальті"
    ]

    print(f"{'Термін':<15} | {'Модель':<10} | {'Найближчі сусіди (Nearest Neighbors)'}")
    print("-" * 100)

    for term in domain_terms:
        print(f"{term:<15} | Word2Vec  | ", end="")
        if term in w2v_model.wv:
            w2v_neighbors = w2v_model.wv.most_similar(term, topn=5)
            print(", ".join([f"{n} ({s:.2f})" for n, s in w2v_neighbors]))
        else:
            print("Слово відсутнє у словнику W2V")

        print(f"{'':<15} | FastText  | ", end="")
        try:
            ft_neighbors = ft_model.wv.most_similar(term, topn=5)
            print(", ".join([f"{n} ({s:.2f})" for n, s in ft_neighbors]))
        except Exception as e:
            print(f"Помилка FastText: {e}")

        print("-" * 100)

analyze_domain_specific_terms(w2v_model, ft_model)

Термін          | Модель     | Найближчі сусіди (Nearest Neighbors)
----------------------------------------------------------------------------------------------------
саундтрек       | Word2Vec  | Слово відсутнє у словнику W2V
                | FastText  | вашингтон (1.00), хяккянен (1.00), хякканен (1.00), ахмат (1.00), вашингтоном (1.00)
----------------------------------------------------------------------------------------------------
енергосистеми   | Word2Vec  | розширення (0.99), посадовець (0.99), складовою (0.99), союзників (0.99), зброї (0.99)
                | FastText  | енергосистему (1.00), енергію (1.00), енергосистемі (1.00), структури (0.99), енергоносіїв (0.99)
----------------------------------------------------------------------------------------------------
санкції         | Word2Vec  | закінчення (0.99), військові (0.98), робота (0.98), агресії (0.98), багатьох (0.98)
                | FastText  | агресії (1.00), дії (0.99), наслідками (0.99), умовами (0.99), по

## Висновки:

Результати Word2Vec виглядають нестабільними. Косинусна близькість 1.00 між словами «пенальті» та «голодомор» або «законопроєкт» та «міф» свідчить про недостатній обсяг навчальних даних і обмежену кількість контекстів. У таких умовах модель фактично фіксує випадкові співзустрічі слів у межах одного речення або новини, що призводить до некоректних семантичних зв’язків.

Результати FastText є більш осмисленими. Модель добре узагальнює морфологічні форми слів (наприклад, «енергосистеми» -> «енергосистему», «законопроєкт» -> «законопроєкту»), а також інколи знаходить семантично близькі поняття (наприклад, «законопроєкт» -> «законодавство»). Водночас для рідкісних слів («саундтрек», «пенальті») якість сусідів залишається низькою через недостатню представленість у корпусі.

Загалом FastText демонструє кращі результати порівняно з Word2Vec, оскільки враховує підсловні n-грами. Це дозволяє моделі ефективно працювати з морфологією та рідкісними словами, навіть якщо вони зустрічаються обмежено. Наприклад, слово «законопроєкт» розкладається на частини «закон-» і «-проєкт», що сприяє зв’язку з іншими спорідненими термінами. Натомість Word2Vec працює лише з цілісними словами і не враховує їхню внутрішню структуру.

# Section 9. 5 “useful / not useful” cases


**Кейс 1:** "енергосистеми" (Доменний термін)

**Сусіди Word2Vec:** потреби (1.00), продуктів (1.00), майбутнього (1.00).

**Сусіди FastText:** енергосистему (1.00), енергію (1.00), енергосистемі (1.00), енергоносіїв (0.99).

**Висновок:** корисно.

**Пояснення:** спостерігається позитивний вплив морфологічної обробки. Модель FastText коректно узагальнює різні словоформи та споріднені терміни (зокрема «енергоносії»), що дозволяє ефективно використовувати вектори для розширення пошукових запитів. Натомість Word2Vec формує менш стабільні зв’язки, часто базуючись лише на локальному контексті.

---

**Кейс 2:** "законопроєкт" (Складне слово)

**Сусіди Word2Vec:** повідомлень (1.00), прокуратури (1.00), срср (1.00).

**Сусіди FastText:** законопроєкту (1.00), відносини (0.99), законодавство (0.99).

**Висновок:** корисно.

**Пояснення:** модель демонструє якісне семантичне сусідство. FastText не лише враховує морфологічну структуру слова, але й знаходить семантично близькі поняття, зокрема «законодавство». Це підтверджує ефективність використання n-грам для роботи зі складними та похідними словами.

---

**Кейс 3:** "пенальті" (Рідкісне слово / Спорт)

**Сусіди Word2Vec:** перевірку (1.00), виграв (1.00), греммі (1.00).

**Сусіди FastText:** лавреата (1.00), диригентом (1.00), гонконгу (1.00).

**Висновок:** некорисно.

**Пояснення:** спостерігається вплив шуму через недостатню представленість слова в корпусі. Word2Vec формує частково логічні, але нестабільні асоціації, тоді як FastText генерує малорелевантні зв’язки через обмежену кількість контекстів. Це свідчить про проблему рідкісних слів (rare words) у навчальних даних.

---

**Кейс 4:** "саундтрек" (Запозичене слово / Домен кіно)

**Сусіди Word2Vec:** Відсутнє у словнику (менше за min_count).

**Сусіди FastText:** сарандос (1.00), вчора (1.00), гандей (1.00).

**Висновок:** некорисно.

**Пояснення:** спостерігається доменна невідповідність та недостатність даних. Через рідкісне використання слова Word2Vec не формує стабільного представлення, тоді як FastText створює вектор на основі субсловних n-грам, що призводить до появи нерелевантних асоціацій.

---

**Кейс 5:** "санкції" (Політичний термін)

**Сусіди Word2Vec:** триває (0.98), гарантії (0.98), сили (0.98).

**Сусіди FastText:** агресії (0.99), акції (0.99), мобілізації (0.99).

**Висновок:** частково корисно/змішаний кейс.

**Пояснення:** Word2Vec формує контекстні зв’язки, пов’язані з подіями та діями, однак вони є семантично слабкими. FastText додатково враховує морфологічну схожість (наприклад, «акції») та тематичну близькість («агресії»), що дозволяє виявити ширше тематичне поле, але іноді призводить до змішування семантичних і поверхневих ознак.

# Section 10. Word2Vec vs FastText comparison


## Підсумкова таблиця порівняння Word2Vec vs FastText

| Word            | Type                   | Word2Vec neighbors                   | FastText neighbors                       | Useful | Comment                                           |
| --------------- | ---------------------- | ------------------------------------ | ---------------------------------------- | ------ | ------------------------------------------------- |
| саундтрек       | domain / rare          | Відсутнє у словнику                  | сарандос, вчора, гандей, омега, хякканен | weak | обидві моделі дають шум через рідкість  |
| енергосистеми   | domain / morph-variant | потреби, продуктів, обороноздатності | енергосистему, енергію, інфраструктури   | useful | FastText краще тримає енергетичний контекст       |
| санкції         | domain / frequent      | гарантії, партнерів, сили            | агресії, мобілізації, дії                | useful | обидві моделі дають політичний контекст           |
| законопроєкт    | domain / frequent      | прокуратури, владу, відносини        | законодавство, судочинства, відносин     | useful | FastText точніше відображає юридичну сферу        |
| пенальті        | domain / frequent      | виграв, правилам, греммі             | дискваліфікацію, кваліфікацію            | weak | Word2Vec шумний, FastText ближчий до спорту       |
| фільм           | frequent / domain      | документальний, нагороду, найкращий  | фільму, фільмі, фільмом                  | useful | FastText стабільніший через морфологію            |
| дискваліфікація | morph-variant          | загрожувала, виступати               | дискваліфікували, дискваліфікації        | useful | FastText добре обробляє словоформи                |
| євробачення     | domain                 | арені, українці, плюс                | постачання, водопостачання               | weak   | обидві моделі дають шум через рідкість            |
| інфляція        | domain                 | оцінка, хвороби, сервіс              | трансляція, оцінка, сервіс               | weak   | слабкі контексти через малу якість корпусу        |
| фейки           | noisy / domain         | опір, рамштайну, європейському       | ілюстрацій, внутрішній, трійки           | partly | FastText трохи стабільніший, але шум все ще є     |
| оприлюднений    | frequent               | подачі, директорів, режисерки        | безкоштовний, благодійний                | weak   | обидві моделі дають нерелевантні сусіди           |
| відновили       | morph-variant          | Відсутнє                             | відносинах, доступні, надходжень         | useful | FastText відновлює форму через підслова           |
| київ            | frequent / entity      | олена, дмитро, сани                  | софія, віталій, наталія                  | partly | обидві моделі змішують з іменами через шум        |
| Phillip Nova    | rare / named entity    | Відсутнє                             | періоду, кінця, віці                     | weak   | FastText не має реального семантичного знання     |
| сценарію        | domain                 | режисерки, парламентської            | films, media, eastfruit                  | partly | обидві моделі дають слабку тематичну стабільність |


| № | Висновок |
|---|----------|
| 1 | Корпус невеликий (837 документів), що обмежує якість embeddings для рідкісних слів |
| 2 | У текстах багато доменних термінів, що дозволяє виявляти тематичні зв’язки |
| 3 | Присутній шум (імена, трансліт, варіації словоформ), який погіршує результати |
| 4 | FastText краще підходить для української мови через обробку морфології та OOV слів |
| 5 | Embeddings дають корисний сигнал, але потребують більшого корпусу та кращого preprocessing |

## Порівняння Word2Vec і FastText

**1. Слова, на яких моделі показали приблизно однакові результати**

На ряді частотних або добре представлених у корпусі слів обидві моделі продемонстрували схожі результати за змістовними сусідами.

Приклади:
* *енергосистеми*: обидві моделі фіксують енергетичний контекст
* *санкції*: обидві моделі пов’язують слово з політичним контекстом (агресія, дії, гарантії)
* *законопроєкт*: FastText і Word2Vec частково узгоджуються в юридичному домені (законодавство, відносини)

**2. Де FastText показав кращі результати**

FastText продемонстрував значну перевагу в обробці рідкісних та морфологічно складних форм.

Приклади:
* Морфологічні варіанти

енергосистеми -> енергосистему / енергосистемі
дискваліфікація -> дискваліфікували
* Рідкісні слова

*саундтрек*

Word2Vec: відсутнє у словнику; FastText: дає хоч і шумні, але векторні наближення
* Невідомі / рідкісні терміни

*Phillip Nova*

Word2Vec: відсутнє; FastText: знаходить схожі контексти через подсловні n-грамми

**3. Де Word2Vec не гірший або навіть кращий**

Word2Vec показує кращу інтерпретованість на стабільних і частотних словах, коли є достатньо контексту в корпусі.

Приклади:
* Часті слова

*фільм*

Word2Vec: документальний, найкращий, нагороду (чіткий семантичний кіно-контекст)
* Стабільні терміни

*пенальті*

Word2Vec: виграв, правилам (більш узагальнено, але стабільно)
* Загальні новинні слова: дають більш “гладкі” тематичні кластери без морфологічного шуму

**4. Підсумковий висновок**

FastText є більш придатною моделлю для даного новинного корпусу, оскільки забезпечує кращу узагальнювальну здатність та стійкість до рідкісних і морфологічно варіативних слів, тоді як Word2Vec демонструє більш “чисті”, але менш повні семантичні зв’язки.

# Section 11. Generate docs/audit_summary_lab9.md

In [ ]:
report = """# Audit Summary — Lab 9 (Word Embeddings: Word2Vec / FastText)
## Корпус і кількість даних

Корпус складається з українських новинних статей різних тематик (політика, спорт, економіка, культура).

Кількість документів: 837

## Які моделі натреновано

Було натреновано дві моделі word embeddings:

1. Word2Vec
2. FastText
## Найсильніші приклади nearest neighbors
1. "фільм"

Word2Vec: документальний, нагороду, найкращий

FastText: фільму, фільмі, фільмом

Пояснення: FastText краще відображає морфологічні форми, Word2Vec — семантичний контекст кіно

2. "дискваліфікація"

Word2Vec: загрожувала, виступати, правилам

FastText: дискваліфікували, дискваліфікації, кваліфікація

Пояснення: FastText значно краще працює зі словоформами

## Найслабші приклади
1. "євробачення"

Word2Vec: арені, українці, плюс

FastText: постачання, водопостачання

Пояснення: слабка семантика через рідкість слова і шум корпусу

2. "інфляція"

Word2Vec: оцінка, хвороби, сервіс

FastText: трансляція, оцінка, сервіс

Пояснення: обидві моделі не формують стабільний економічний контекст

3. "оприлюднений"

Word2Vec: подачі, директорів, режисерки

FastText: безкоштовний, благодійний

Пояснення: слово недостатньо контекстно представлене в корпусі

## Осмислені доменні терміни
1. "енергосистеми"

Word2Vec: потреби, продуктів, майбутнього

FastText: енергосистему, енергію, енергосистемі, енергоносіїв

Пояснення: FastText чітко утримує енергетичний домен і морфологію

2. "законопроєкт"

Word2Vec: прокуратури, владу, повідомлень

FastText: законопроєкту, законодавство, судочинства

Пояснення: FastText краще відображає юридичний контекст

## Де FastText показав перевагу

FastText демонструє кращі результати завдяки використанню subword-інформації:

* Морфологічні варіанти

"енергосистеми" -> енергосистему / енергосистемі,
дискваліфікація -> дискваліфікували
* Рідкісні слова

"саундтрек" — відсутній у Word2Vec, FastText формує хоча б наближені вектори
* Невідомі / named entities

"Phillip Nova" — відсутній у Word2Vec, FastText дає слабкі, але можливі наближення

## Де виграш мінімальний
1. "саундтрек"

Word2Vec: відсутнє у словнику

FastText: сарандос, вчора, гандей, омега, хякканен

Пояснення: це пов’язано з низькою частотою слова в корпусі та відсутністю достатнього контексту для формування якісного вектора

2. "пенальті"

Word2Vec: виграв, правилам, греммі

FastText: дискваліфікацію, кваліфікацію

Пояснення: моделі плутають спортивні терміни через перетин контекстів і обмежену представленість вузькоспеціалізованої лексики в корпусі

## Чи варто використовувати embeddings далі

Так, embeddings варто використовувати далі, оскільки вони дозволяють виявляти семантичні зв’язки між словами та покращують якість аналізу текстів. У даному корпусі моделі показали здатність відображати тематичні залежності, особливо для частотних і доменних термінів. FastText є більш ефективним для української мови завдяки обробці морфологічних варіантів і рідкісних слів. Водночас результати можуть бути нестабільними для малочастотних або шумних даних. Для підвищення якості доцільно збільшити обсяг корпусу та покращити попередню обробку тексту. Таким чином, embeddings є корисним інструментом, але потребують оптимізації для отримання кращих результатів.
"""

with open("audit_summary_lab9.md", "w", encoding="utf-8") as f:
    f.write(report)